<a href="https://colab.research.google.com/github/kevinn78/salud_mental/blob/main/Proyecto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import pandas as pd
import numpy as np
import os
from google.colab import drive

# 1. Montar Drive y definir rutas (Módulo 1.4)
drive.mount('/content/drive')
path_raiz = '/content/drive/MyDrive/semestre 5/PROGRAMACION PARA LA CIENCIA DE DATOS/'
path_raw = path_raiz + 'data/raw/'
path_processed = path_raiz + 'data/processed/'

if not os.path.exists(path_processed):
    os.makedirs(path_processed)

# 2. Cargar Dataset Original
nombre_archivo = 'Dataset_salud_mental.zip'
path_completo = os.path.join(path_raw, nombre_archivo)

print("--- FASE 1: CARGA DE DATOS ---")
df_mental = pd.read_csv(path_completo, compression='zip')

print(f"📊 Registros iniciales: {len(df_mental)} | Columnas: {len(df_mental.columns)}")
print("\n🔍 Diagnóstico de Nulos (Huecos):")
print(df_mental.isnull().sum()[df_mental.isnull().sum() > 0]) # Solo muestra columnas con nulos

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
--- FASE 1: CARGA DE DATOS ---
📊 Registros iniciales: 10000 | Columnas: 16

🔍 Diagnóstico de Nulos (Huecos):
Country         2071
Country_Code    1283
dtype: int64


In [8]:
print("--- FASE 2: LIMPIEZA DE DATOS (DATA CLEANING) ---")

# COPIA DE SEGURIDAD: Siempre trabajamos sobre una copia para no dañar el original
df_clean = df_mental.copy()

# NIVEL 1: Tratamiento de Duplicados
duplicados = df_clean.duplicated().sum()
print(f"1. Duplicados encontrados: {duplicados}")
df_clean = df_clean.drop_duplicates()

# NIVEL 2: Imputación de Nulos (En vez de borrar, rellenamos inteligentemente)
# Rellenamos los países vacíos con la categoría "No especificado"
df_clean['Country'] = df_clean['Country'].fillna('No especificado')
df_clean['Country_Code'] = df_clean['Country_Code'].fillna('ND')

# Estandarización de texto (Mayúsculas y minúsculas)
df_clean['Country'] = df_clean['Country'].str.strip().str.title()
df_clean['Country_Code'] = df_clean['Country_Code'].str.strip().str.upper()

# NIVEL 3: Tratamiento de Outliers (Valores atípicos)
# Ejemplo: Las horas de sueño lógicamente deberían estar entre 2 y 14 horas máximo.
# Si alguien puso 50 horas, lo "capeamos" (limitamos) al máximo lógico.
df_clean['sleep_hours'] = np.clip(df_clean['sleep_hours'], a_min=2, a_max=14)

print("✅ Limpieza completada. Verificación de nulos post-limpieza:")
print(df_clean.isnull().sum().sum(), "nulos restantes.")

--- FASE 2: LIMPIEZA DE DATOS (DATA CLEANING) ---
1. Duplicados encontrados: 0
✅ Limpieza completada. Verificación de nulos post-limpieza:
0 nulos restantes.


In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

print("--- FASE 3: TRANSFORMACIÓN (PIPELINE) ---")

# 1. Definimos qué columnas son numéricas y cuáles son texto (categorías)
# Elegimos algunas clave para el modelo
columnas_numericas = ['age', 'stress_level', 'sleep_hours', 'depression_score']
columnas_categoricas = ['gender', 'work_environment']

# 2. El "Director de Orquesta" (ColumnTransformer)
# - A los números les aplicamos StandardScaler (Media 0, Desviación 1)
# - A los textos les aplicamos OneHotEncoder (Ceros y Unos)
preprocesador = ColumnTransformer(transformers=[
    ('num', StandardScaler(), columnas_numericas),
    ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), columnas_categoricas)
])

# 3. La "Tubería" (Pipeline)
pipeline_final = Pipeline(steps=[
    ('prep', preprocesador)
])

# 4. Ejecución Mágica (fit_transform)
datos_transformados = pipeline_final.fit_transform(df_clean)

print("✅ Datos transformados exitosamente en una matriz matemática.")
print(f"Tamaño de la nueva matriz para la IA: {datos_transformados.shape}")

# --- FASE 4: GUARDADO (EXPORTACIÓN) ---
# Guardamos el dataset limpio (df_clean) para hacer nuestros gráficos después
archivo_salida = os.path.join(path_processed, 'Mental_Health_Clean.csv')
df_clean.to_csv(archivo_salida, index=False)

print(f"\n📂 Archivo limpio y listo guardado en: {archivo_salida}")
print("--- PROCESO INDUSTRIAL COMPLETADO ---")

--- FASE 3: TRANSFORMACIÓN (PIPELINE) ---
✅ Datos transformados exitosamente en una matriz matemática.
Tamaño de la nueva matriz para la IA: (10000, 11)

📂 Archivo limpio y listo guardado en: /content/drive/MyDrive/semestre 5/PROGRAMACION PARA LA CIENCIA DE DATOS/data/processed/Mental_Health_Clean.csv
--- PROCESO INDUSTRIAL COMPLETADO ---
